# S.T.I.T.C.H — Floorplan Segmentation Training
Run each cell top to bottom.

In [ ]:
# CELL 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CELL 2 — Check GPU
!nvidia-smi

Mon Jul 27 14:31:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             31W /   70W |    1025MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# CELL 3 — Unzip dataset from Drive
import os

ZIP_PATH = '/content/drive/MyDrive/dataset.zip'
EXTRACT_PATH = '/content/'

print('Unzipping dataset...')
!unzip -q {ZIP_PATH} -d {EXTRACT_PATH}
print('Done!')
print('Total images:', len(os.listdir('/content/dataset/images')))
print('Total masks: ', len(os.listdir('/content/dataset/masks')))

Unzipping dataset...
replace /content/dataset/images/cc5k_train_5715_png.rf.c2478ffa6163af928c067325c909979a.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
A
A
Done!
Total images: 10350
Total masks:  10350


In [ ]:
# CELL 4 — Install dependencies
!pip install tqdm opencv-python-headless albumentations segmentation-models-pytorch -q

in the CEll 5 either use 5.1 or use 5.2 do not use both of them together then the model will not work

if use 5.1 then use 7.1 because that is wat you neet to get the Unet model

but if you use the 5.2 then you have to use the 7.2 training code so then you can get the pretrained restnet model in the Unet

the code to us ethem both are uploaded in github and are in the same file called "predict_tiled.py" (they are devided in segments the opne above is the one for the Unet model to parse and the 2nd one is to parse the unet model with combined loss (BCE + DICE loss))

In [ ]:
# CELL 5.1 — Model definition Unet (unchanged)
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DoubleConv(3, 64)
        self.p1 = nn.MaxPool2d(2)
        self.d2 = DoubleConv(64, 128)
        self.p2 = nn.MaxPool2d(2)
        self.d3 = DoubleConv(128, 256)
        self.p3 = nn.MaxPool2d(2)
        self.b  = DoubleConv(256, 512)
        self.u3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.c3 = DoubleConv(512, 256)
        self.u2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.c2 = DoubleConv(256, 128)
        self.u1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.c1 = DoubleConv(128, 64)
        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(self.p1(d1))
        d3 = self.d3(self.p2(d2))
        b  = self.b(self.p3(d3))
        u3 = self.c3(torch.cat([self.u3(b), d3], dim=1))
        u2 = self.c2(torch.cat([self.u2(u3), d2], dim=1))
        u1 = self.c1(torch.cat([self.u1(u2), d1], dim=1))
        return self.out(u1)

print('✅ Model defined')

✅ Model defined


In [ ]:
# CELL 5.2 — Model: UNet with pretrained ResNet34 encoder
# segmentation-models-pytorch gives us a pretrained ImageNet encoder for free
# walls, edges, and corners transfer well from ImageNet — much better starting point
# than training from scratch
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

def get_model():
    model = smp.Unet(
        encoder_name='resnet34',      # pretrained ResNet34 encoder
        encoder_weights='imagenet',   # start from ImageNet weights, not random
        in_channels=3,
        classes=1,
    )
    return model

# sanity check
test_model = get_model()
test_input = torch.randn(1, 3, 256, 256)
test_out   = test_model(test_input)
print(f'Model output shape: {test_out.shape}')  # should be [1, 1, 256, 256]
print(' Model defined — ResNet34 encoder with ImageNet weights')
del test_model, test_input, test_out

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Model output shape: torch.Size([1, 1, 256, 256])
 Model defined — ResNet34 encoder with ImageNet weights


In [ ]:
# CELL 6 — Dataset WITH augmentation + stroke masks + hard negatives
import os
import cv2
import numpy as np
import random
import albumentations as A
import torch
from torch.utils.data import Dataset, DataLoader


train_transform = A.Compose([
    A.Rotate(limit=180, p=0.8),
    A.ElasticTransform(alpha=120, sigma=6, p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
], additional_targets={'mask': 'mask'})


def add_hard_negatives(img, mask):
    """Stamp synthetic furniture shapes onto image, mark them as background."""
    h, w = mask.shape
    for _ in range(random.randint(2, 6)):
        cx = random.randint(20, w - 20)
        cy = random.randint(20, h - 20)
        sz = random.randint(8, 30)
        temp = np.zeros((h, w), np.uint8)
        if random.random() < 0.5:
            cv2.rectangle(temp, (cx - sz, cy - sz), (cx + sz, cy + sz), 1, -1)
        else:
            cv2.circle(temp, (cx, cy), sz, 1, -1)
        # Only zero out pixels that aren't already real walls
        mask[temp == 1] = 0
    return img, mask


class FloorplanDataset(Dataset):
    def __init__(self, img_dir, mask_dir, augment=False):
        self.img_names = sorted(os.listdir(img_dir))
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir
        self.augment   = augment

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        name = self.img_names[idx]

        img = cv2.imread(os.path.join(self.img_dir, name))
        if img is None:
            return self.__getitem__(0)
        img = cv2.resize(img, (256, 256))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(os.path.join(self.mask_dir, name), cv2.IMREAD_UNCHANGED)
        if len(mask.shape) == 3:
            mask = mask[:, :, 0]
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0).astype(np.uint8)

        # Hard negatives — 50% of samples
        if self.augment and random.random() < 0.5:
            img, mask = add_hard_negatives(img, mask)

        # Albumentations augmentation
        if self.augment:
            aug   = train_transform(image=img, mask=mask)
            img   = aug['image']
            mask  = aug['mask']

        img  = img.astype(np.float32) / 255.0
        img  = torch.from_numpy(img).permute(2, 0, 1)
        mask = mask.astype(np.float32)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return img, mask

print(' Dataset defined (augmentation ON, hard negatives ON)')

 Dataset defined (augmentation ON, hard negatives ON)


In [ ]:
# CELL 7.1 — Training (BCE loss + unet model)
import torch.optim as optim
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

dataset = FloorplanDataset('/content/dataset/images', '/content/dataset/masks', augment=True)
print(f'Total images: {len(dataset)}')

loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
print(f'Total batches per epoch: {len(loader)}')

model     = UNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# pos_weight=3.0 — walls are sparse pixels; penalise missing them 3x more than false positives
# this is the main fix for single-line walls being ignored during training
loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([3.0]).to(device)
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5
)

epochs    = 15
best_loss = float('inf')

print('\nStarting training...')

for epoch in range(epochs):
    model.train()
    total_loss = 0
    loop = tqdm(loader, desc=f'Epoch {epoch+1}/{epochs}')

    for imgs, masks in loop:
        imgs  = imgs.to(device)
        masks = masks.to(device)
        preds = model(imgs)
        loss  = loss_fn(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        loop.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = total_loss / len(loader)
    print(f'Epoch {epoch+1} — Avg Loss: {avg_loss:.4f}')

    scheduler.step(avg_loss)

    torch.save(model.state_dict(), '/content/unet.pth')

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), '/content/unet_best.pth')
        print(f'   Best model updated (loss: {best_loss:.4f})')

print(f'\n TRAINING COMPLETE — best loss: {best_loss:.4f}')

Using device: cuda
Total images: 10350
Total batches per epoch: 2588

Starting training...


Epoch 1/15:  52%|█████▏    | 1353/2588 [04:33<04:09,  4.95it/s, loss=0.3824]

In [ ]:
# CELL 7.2 — Training (BCE loss + Dice loss + unet)
import torch.optim as optim
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

dataset = FloorplanDataset('/content/dataset/images', '/content/dataset/masks', augment=True)
print(f'Total images: {len(dataset)}')

loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
print(f'Total batches per epoch: {len(loader)}')

model     = get_model().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# BCE — penalises missing wall pixels 3x more than false positives
bce_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([3.0]).to(device)
)

def dice_loss(pred, target, smooth=1.0):
    """Dice loss — optimises directly for mask overlap, not pixel accuracy.
    BCE cares about each pixel equally; Dice cares about getting the whole
    wall shape right. Together they cover both."""
    pred   = torch.sigmoid(pred)
    flat_p = pred.view(-1)
    flat_t = target.view(-1)
    intersection = (flat_p * flat_t).sum()
    return 1 - (2 * intersection + smooth) / (flat_p.sum() + flat_t.sum() + smooth)

def combined_loss(pred, target):
    """50/50 BCE + Dice — standard combination for binary segmentation."""
    return 0.5 * bce_fn(pred, target) + 0.5 * dice_loss(pred, target)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5
)

epochs    = 15
best_loss = float('inf')

print('\nStarting training...')

for epoch in range(epochs):
    model.train()
    total_loss = 0
    loop = tqdm(loader, desc=f'Epoch {epoch+1}/{epochs}')

    for imgs, masks in loop:
        imgs  = imgs.to(device)
        masks = masks.to(device)
        preds = model(imgs)
        loss  = combined_loss(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        loop.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = total_loss / len(loader)
    print(f'Epoch {epoch+1} — Avg Loss: {avg_loss:.4f}')

    scheduler.step(avg_loss)

    torch.save(model.state_dict(), '/content/unet.pth')

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), '/content/unet_best.pth')
        print(f'   Best model updated (loss: {best_loss:.4f})')

print(f'\n TRAINING COMPLETE — best loss: {best_loss:.4f}')


Using device: cuda
Total images: 10350
Total batches per epoch: 2588

Starting training...


Epoch 1/1: 100%|██████████| 2588/2588 [03:22<00:00, 12.76it/s, loss=0.4739]


Epoch 1 — Avg Loss: 0.5312
   Best model updated (loss: 0.5312)

 TRAINING COMPLETE — best loss: 0.5312


In [ ]:
# CELL 8 — Save both weights to Google Drive
import shutil
shutil.copy('/content/unet.pth',      '/content/drive/MyDrive/unet.pth')
shutil.copy('/content/unet_best.pth', '/content/drive/MyDrive/unet_best.pth')
print('✅ Both weights saved to Google Drive!')
print('   unet.pth      — final epoch')
print('   unet_best.pth — best loss epoch')

✅ Both weights saved to Google Drive!
   unet.pth      — final epoch
   unet_best.pth — best loss epoch
